# Setup

In [1]:
import duckdb

con = duckdb.connect("amazon_fashion.duckdb")

con.execute("""
    CREATE TABLE IF NOT EXISTS amazon_fashion AS
    SELECT *
    FROM read_json_auto('../datasets/meta_Amazon_Fashion.jsonl');
""")

con.close()

In [2]:
import duckdb
import pandas as pd

con = duckdb.connect("amazon_fashion.duckdb")

# 1. Sanity Checks

In [3]:
total_rows = con.execute("""
    SELECT count(*) FROM amazon_fashion
""").fetchone()[0]

print("Total rows:", total_rows)

Total rows: 826108


# Preview

In [4]:
preview_df = con.execute("""
    SELECT * FROM amazon_fashion LIMIT 10
""").df()

print(preview_df.head())

    main_category                                              title  \
0  AMAZON FASHION  YUEDGE 5 Pairs Men's Moisture Control Cushione...   
1  AMAZON FASHION  DouBCQ Women's Palazzo Lounge Wide Leg Casual ...   
2  AMAZON FASHION  Pastel by Vivienne Honey Vanilla Girls' Trapez...   
3  AMAZON FASHION                                   Mento Streamtail   
4  AMAZON FASHION  RONNOX Women's 3-Pairs Bright Colored Calf Com...   

   average_rating  rating_number  \
0             4.6             16   
1             4.1              7   
2             4.3             11   
3             2.0              1   
4             4.3           3032   

                                            features  \
0                                                 []   
1                 [Drawstring closure, Machine Wash]   
2                   [Zipper closure, Hand Wash Only]   
3  [Thermoplastic Rubber sole, High Density Premi...   
4  [Pull On closure, Size Guide: "S" fits calf 10...   

             

Missing / null analysis

In [5]:
nulls_df = con.execute("""
    SELECT
        count(*) AS total,
        count(main_category) AS main_category_non_null,
        count(title) AS title_non_null,
        count(price) AS price_non_null,
        count(average_rating) AS rating_non_null
    FROM amazon_fashion
""").df()

print(nulls_df)

    total  main_category_non_null  title_non_null  price_non_null  \
0  826108                  826108          826108           50249   

   rating_non_null  
0           826108  


Numeric EDA (price & ratings)

In [6]:
price_stats = con.execute("""
    SELECT
        MIN(price) AS min_price,
        MAX(price) AS max_price,
        AVG(price) AS avg_price,
        MEDIAN(price) AS median_price
    FROM amazon_fashion
    WHERE price IS NOT NULL
""").df()

print(price_stats)

   min_price  max_price  avg_price  median_price
0       0.01    13000.0  40.795929         19.89


Rating Distribution

In [7]:
rating_dist = con.execute("""
    SELECT
        average_rating,
        count(*) AS cnt
    FROM amazon_fashion
    WHERE average_rating IS NOT NULL
    GROUP BY average_rating
    ORDER BY average_rating
""").df()

print(rating_dist.head())

   average_rating    cnt
0             1.0  29905
1             1.1      6
2             1.2     53
3             1.3    372
4             1.4    835


# Category analysis

In [8]:
main_category_df = con.execute("""
    SELECT
        main_category,
        count(*) AS cnt
    FROM amazon_fashion
    GROUP BY main_category
    ORDER BY cnt DESC
    LIMIT 20
""").df()

print(main_category_df)

    main_category     cnt
0  AMAZON FASHION  826108


Flattened categories (JSON[])

In [9]:
categories_df = con.execute("""
    SELECT
        json_extract_string(cat, '$') AS category,
        count(*) AS cnt
    FROM amazon_fashion,
    UNNEST(categories) AS t(cat)
    GROUP BY category
    ORDER BY cnt DESC
    LIMIT 30
""").df()

print(categories_df.head())

Empty DataFrame
Columns: [category, cnt]
Index: []


5️⃣ Text EDA

In [10]:
title_len_df = con.execute("""
    SELECT
        min(length(title)) AS min_len,
        max(length(title)) AS max_len,
        avg(length(title)) AS avg_len
    FROM amazon_fashion
    WHERE title IS NOT NULL
""").df()

print(title_len_df)

   min_len  max_len    avg_len
0        0     1536  92.916973


In [11]:
desc_len_df = con.execute("""
    SELECT
        avg(length(array_to_string(description, ' '))) AS avg_desc_len
    FROM amazon_fashion
    WHERE description IS NOT NULL
""").df()

print(desc_len_df)

   avg_desc_len
0    531.997858


6️⃣ Images & videos coverage

In [12]:
images_df = con.execute("""
    SELECT
        avg(array_length(images)) AS avg_images,
        max(array_length(images)) AS max_images
    FROM amazon_fashion
    WHERE images IS NOT NULL
""").df()

print(images_df)

   avg_images  max_images
0    4.553579          31


In [13]:
images_coverage_df = con.execute("""
    SELECT
        count(*) FILTER (WHERE images IS NOT NULL AND array_length(images) > 0) AS with_images,
        count(*) AS total
    FROM amazon_fashion
""").df()

print(images_coverage_df)

   with_images   total
0       826107  826108


Video coverage

In [14]:
videos_df = con.execute("""
    SELECT
        count(*) FILTER (WHERE videos IS NOT NULL AND array_length(videos) > 0) AS with_videos,
        count(*) AS total
    FROM amazon_fashion
""").df()

print(videos_df)

   with_videos   total
0        52107  826108


7️⃣ Store / brand analysis

In [15]:
store_df = con.execute("""
    SELECT
        store,
        count(*) AS cnt
    FROM amazon_fashion
    WHERE store IS NOT NULL
    GROUP BY store
    ORDER BY cnt DESC
    LIMIT 20
""").df()

print(store_df)

           store   cnt
0           Nike  5154
1          Romwe  4150
2    GRACE KARIN  3833
3        Verdusa  3723
4        Floerns  3441
5          SheIn  3066
6         Disney  2849
7        Ekouaer  2764
8    SweatyRocks  2661
9        Milumia  2186
10        Zeagoo  2165
11       Generic  2093
12     Allegra K  2014
13       DOUBLJU  1855
14    MakeMeChic  1773
15        adidas  1743
16      SOLY HUX  1698
17      COOFANDY  1690
18    Lands' End  1660
19  Michael Kors  1385


8️⃣ Details MAP analysis (keys frequency)

In [16]:
details_keys_df = con.execute("""
    SELECT
        key,
        count(*) AS cnt
    FROM amazon_fashion,
    UNNEST(map_keys(details)) AS t(key)
    GROUP BY key
    ORDER BY cnt DESC
""").df()

print(details_keys_df.head())

                               key     cnt
0             Date First Available  797051
1               Package Dimensions  528478
2                Item model number  371915
3  Is Discontinued By Manufacturer  295500
4               Product Dimensions  160469


9️⃣ Bought-together coverage

In [17]:
bought_together_df = con.execute("""
    SELECT
        count(*) FILTER (WHERE bought_together IS NOT NULL) AS has_bt,
        count(*) AS total
    FROM amazon_fashion
""").df()

print(bought_together_df)

   has_bt   total
0       0  826108


🔟 Data quality checks

In [18]:
bad_price_df = con.execute("""
    SELECT *
    FROM amazon_fashion
    WHERE price < 0 OR price > 10000
    LIMIT 20
""").df()

print(bad_price_df)

    main_category                                              title  \
0  AMAZON FASHION  Breitling Navitimer Automatic Chronograph Blac...   
1  AMAZON FASHION  Ulysse Nardin Marine Regatta Stainless Steel/B...   
2  AMAZON FASHION  Omega Seamaster Ploprof Automatic Mens Watch 2...   

   average_rating  rating_number features description    price  \
0             2.0              1       []          []  13000.0   
1             5.0              1       []          []  12111.5   
2             5.0              1       []          []  11340.0   

                                              images videos          store  \
0  [{'thumb': 'https://m.media-amazon.com/images/...     []      Breitling   
1  [{'thumb': 'https://m.media-amazon.com/images/...     []  Ulysse Nardin   
2  [{'thumb': 'https://m.media-amazon.com/images/...     []          Omega   

  categories                                            details parent_asin  \
0         []  {'Is Discontinued By Manufacturer': 'No'

Ratings without rating count

In [19]:
rating_mismatch_df = con.execute("""
    SELECT *
    FROM amazon_fashion
    WHERE average_rating IS NOT NULL AND rating_number IS NULL
    LIMIT 20
""").df()

print(rating_mismatch_df)

Empty DataFrame
Columns: [main_category, title, average_rating, rating_number, features, description, price, images, videos, store, categories, details, parent_asin, bought_together]
Index: []


In [20]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(details->>'Department') AS department_present,
    COUNT(*) - COUNT(details->>'Department') AS department_missing
FROM amazon_fashion;
"""

print(con.execute(query).fetchdf())

   total_rows  department_present  department_missing
0      826108              119646              706462


1️⃣ How many unique Departments?

In [21]:
con.execute("""
    SELECT
        COUNT(DISTINCT details->>'Department') AS unique_departments
    FROM amazon_fashion
    WHERE details->>'Department' IS NOT NULL
""").df()

,unique_departments
0,1202


2️⃣ Top Departments by product count

In [22]:
dept_counts = con.execute("""
    SELECT
        details->>'Department' AS department,
        COUNT(*) AS product_count
    FROM amazon_fashion
    WHERE details->>'Department' IS NOT NULL
    GROUP BY department
    ORDER BY product_count DESC
    LIMIT 20
""").df()

print(dept_counts)


      department  product_count
0         Womens          26964
1         womens          25868
2           Mens          10324
3           mens           9207
4   Unisex-adult           5932
5   unisex-adult           5644
6          Girls           4245
7          girls           3982
8          Women           3499
9   Unisex Adult           2693
10          Boys           2192
11          boys           2161
12  Unisex-child           2000
13  unisex-child           1991
14        Unisex           1865
15           Men           1402
16    baby-girls            684
17    Baby-girls            670
18     baby-boys            534
19     Baby-boys            528


3️⃣ Department coverage (% of catalog)

In [23]:
con.execute("""
    SELECT
        100.0 * COUNT(details->>'Department') / COUNT(*) AS department_coverage_pct
    FROM amazon_fashion
""").df()

,department_coverage_pct
0,14.483094


4️⃣ Price statistics by Department

In [24]:
dept_price_stats = con.execute("""
    SELECT
        details->>'Department' AS department,
        COUNT(*) AS n_products,
        MIN(price) AS min_price,
        MAX(price) AS max_price,
        AVG(price) AS avg_price,
        MEDIAN(price) AS median_price
    FROM amazon_fashion
    WHERE price IS NOT NULL
      AND details->>'Department' IS NOT NULL
    GROUP BY department
    HAVING COUNT(*) >= 50
    ORDER BY avg_price DESC
""").df()

print(dept_price_stats.head())

ConversionException: Conversion Error: Unimplemented type for cast (MAP(VARCHAR, VARCHAR) -> BOOLEAN) when casting from source column details

LINE 11:       AND details->>'Department' IS NOT NULL
                   ^

In [ ]:
con.execute("""
    SELECT
        details->>'Department' AS department,
        AVG(length(array_to_string(description, ' '))) AS avg_desc_length
    FROM amazon_fashion
    WHERE description IS NOT NULL
      AND details->>'Department' IS NOT NULL
    GROUP BY department
    ORDER BY avg_desc_length DESC
""").df()

ConversionException: Conversion Error: Unimplemented type for cast (MAP(VARCHAR, VARCHAR) -> BOOLEAN) when casting from source column details

LINE 7:       AND details->>'Department' IS NOT NULL
                  ^

In [27]:
image_df = con.execute("""
    SELECT images FROM amazon_fashion LIMIT 10
""").df()

print(image_df.head())

                                              images
0  [{'thumb': 'https://m.media-amazon.com/images/...
1  [{'thumb': 'https://m.media-amazon.com/images/...
2  [{'thumb': 'https://m.media-amazon.com/images/...
3  [{'thumb': 'https://m.media-amazon.com/images/...
4  [{'thumb': 'https://m.media-amazon.com/images/...
